# Experiment: Oracle Beta Search

Objective:
- Use the same six application-style scenarios as the cross-method notebook.
- Initialize beta from a pilot-only rule, without the discrete scan.
- Run Optuna HPO over beta multiplier and swap size to obtain an oracle scenario-specific beta.
- Save the best oracle beta and compare it against the pilot-only initialization.

In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    build_beta_initialization,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    run_named_mcmc_checkpoint_study,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
)

pd.set_option("display.max_columns", 100)
project_root

In [ ]:
import json
import numpy as np
import optuna
from optuna.samplers import TPESampler

## Configuration

`ESTIMATION_POINTS` are the final evaluation checkpoints for the pilot-initialized and oracle-selected configurations.  
The Optuna objective itself is evaluated at `HPO_CHECKPOINT`, which is treated as an extra pre-production calibration budget and is not part of the article x-axis budget.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_beta_notebook"

SCENARIO_GROUP = "threshold_suite"
SCENARIO_KEYS_OVERRIDE = [
    "gwas_additive_score_ultra_n60",
    "poisson_diffmeans_hep_ultra_n200",
    "gwas_additive_score_sig_n60",
    "poisson_diffmeans_hep_sig_n200",
    "gwas_additive_score_above_n60",
    "poisson_diffmeans_hep_above_n200",
]
ESTIMATION_POINTS = (5_000_000,) if not FAST_MODE else (200_000,)
HPO_CHECKPOINT = max(ESTIMATION_POINTS)
HPO_N_TRIALS = 30 if not FAST_MODE else 5
HPO_TRIAL_REPEATS = 3 if not FAST_MODE else 1
HPO_OBJECTIVE_METRIC = "rmse"
HPO_BETA_MULTIPLIER_LOW = 0.25
HPO_BETA_MULTIPLIER_HIGH = 4.0
HPO_SWAP_CHOICES_DEFAULT = (1, 2, 3, 4, 6, 8)
HPO_SWAP_CHOICES_BY_SETTING = {
    "gwas_threshold_suite": (1, 2, 3, 4),
    "hep_threshold_suite": (2, 4, 6, 8, 10),
}
DEFAULT_BASELINE_SWAP_BY_SETTING = {
    "gwas_threshold_suite": 2,
    "hep_threshold_suite": 6,
}
FINAL_EVAL_REPEATS = 5 if not FAST_MODE else 2
FINAL_EVAL_N_JOBS = min(FINAL_EVAL_REPEATS, os.cpu_count() or 1)
MIN_TAIL_STATES = 2
BASE_SEED = 54_321

init_cfg = MCMCWorkflowConfig(
    use_true_p0_for_q_target=False,
    p0_guess=1e-8,
    pilot_samples=20_000 if not FAST_MODE else 1_000,
    scale_method="sd",
    beta_max_init=1e6,
    chains=2,
    thin=1,
    estimate_variance=False,
    proposal_size=1,
)

hpo_eval_cfg = BetaSweepStudyConfig(
    estimation_points=(HPO_CHECKPOINT,),
    repeats=HPO_TRIAL_REPEATS,
    beta_multipliers=(1.0,),
    chains=2,
    thin=1,
    estimate_variance=False,
    proposal_size=1,
    base_seed=BASE_SEED,
    n_jobs=1,
)

final_eval_cfg = BetaSweepStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=FINAL_EVAL_REPEATS,
    beta_multipliers=(1.0,),
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_size=1,
    base_seed=BASE_SEED + 500_000,
    n_jobs=FINAL_EVAL_N_JOBS,
)

def reference_p0_for_scenario(scenario) -> float:
    threshold = scenario.extra.get("known_significance_threshold")
    if threshold is not None:
        threshold_f = float(threshold)
        if threshold_f == threshold_f and threshold_f > 0.0:
            return threshold_f
    return float(scenario.exact_p)


def swap_choices_for_scenario(scenario) -> tuple[int, ...]:
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key in HPO_SWAP_CHOICES_BY_SETTING:
        return tuple(int(v) for v in HPO_SWAP_CHOICES_BY_SETTING[setting_key])
    return tuple(int(v) for v in HPO_SWAP_CHOICES_DEFAULT)


def baseline_swap_for_scenario(scenario) -> int:
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key in DEFAULT_BASELINE_SWAP_BY_SETTING:
        return int(DEFAULT_BASELINE_SWAP_BY_SETTING[setting_key])
    return int(swap_choices_for_scenario(scenario)[0])


def trial_objective_value(summary_row: dict, metric: str) -> float:
    metric_val = float(summary_row[metric])
    if not np.isfinite(metric_val):
        return float("inf")
    return metric_val


print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_GROUP": SCENARIO_GROUP,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "HPO_CHECKPOINT": HPO_CHECKPOINT,
    "HPO_N_TRIALS": HPO_N_TRIALS,
    "HPO_TRIAL_REPEATS": HPO_TRIAL_REPEATS,
    "HPO_OBJECTIVE_METRIC": HPO_OBJECTIVE_METRIC,
    "HPO_BETA_MULTIPLIER_LOW": HPO_BETA_MULTIPLIER_LOW,
    "HPO_BETA_MULTIPLIER_HIGH": HPO_BETA_MULTIPLIER_HIGH,
    "HPO_SWAP_CHOICES_BY_SETTING": HPO_SWAP_CHOICES_BY_SETTING,
    "FINAL_EVAL_REPEATS": FINAL_EVAL_REPEATS,
    "FINAL_EVAL_N_JOBS": FINAL_EVAL_N_JOBS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
    "REFERENCE_P0_MODE": "known_significance_threshold_else_exact_p",
}, indent=2))

## Load Scenarios

In [ ]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None if SCENARIO_KEYS_OVERRIDE is not None else SCENARIO_GROUP,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "beta_diag") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
        "family": s.portfolio.get("family"),
        "rarity_band": s.portfolio.get("rarity_band"),
        "difficulty": s.portfolio.get("expected_difficulty"),
    }
    for s in scenarios
])

## Run Oracle Beta HPO

In [ ]:
oracle_results = {}

for scenario_idx, scenario in enumerate(scenarios):
    scenario_seed = BASE_SEED + 50_000 * scenario_idx
    p0_reference = reference_p0_for_scenario(scenario)
    init_payload = build_beta_initialization(
        scenario.problem,
        scenario.exact_p,
        init_cfg,
        seed=scenario_seed,
        p0_reference=p0_reference,
    )
    beta_init = float(init_payload["beta0_laplace"])
    sigma_t = float(init_payload["sigma_t"])
    swap_choices = swap_choices_for_scenario(scenario)
    baseline_swap = baseline_swap_for_scenario(scenario)
    trial_rows = []

    print(json.dumps({
        "scenario": scenario.key,
        "exact_p": scenario.exact_p,
        "application_setting_key": scenario.extra.get("application_setting_key"),
        "known_significance_threshold": scenario.extra.get("known_significance_threshold"),
        "reference_p0": p0_reference,
        "beta0_formula": init_payload["beta0_formula"],
        "beta0_laplace": beta_init,
        "q_target": init_payload["q_target"],
        "sigma_t": sigma_t,
        "swap_choices": swap_choices,
        "baseline_swap": baseline_swap,
    }, indent=2))

    def objective(trial):
        beta_multiplier = float(
            trial.suggest_float(
                "beta_multiplier",
                HPO_BETA_MULTIPLIER_LOW,
                HPO_BETA_MULTIPLIER_HIGH,
                log=True,
            )
        )
        proposal_size = int(trial.suggest_categorical("proposal_size", list(swap_choices)))
        beta = float(beta_init * beta_multiplier)
        trial_eval = run_named_mcmc_checkpoint_study(
            scenario.problem,
            scenario.exact_p,
            config_specs=[
                {
                    "label": "oracle_trial",
                    "config_id": f"trial_{trial.number}",
                    "beta": beta,
                    "proposal_size": proposal_size,
                    "source": "oracle_hpo",
                }
            ],
            sigma_t=sigma_t,
            estimation_points=(HPO_CHECKPOINT,),
            repeats=HPO_TRIAL_REPEATS,
            base_seed=BASE_SEED + 1_000_000 * scenario_idx + 10_000 * trial.number,
            template_cfg=hpo_eval_cfg,
            n_jobs=int(hpo_eval_cfg.n_jobs),
        )
        summary_row = dict(pd.DataFrame(trial_eval["summary"]).iloc[0])
        objective_value = trial_objective_value(summary_row, HPO_OBJECTIVE_METRIC)
        trial_payload = {
            "trial_number": int(trial.number),
            "scenario": scenario.key,
            "beta_multiplier": beta_multiplier,
            "beta": beta,
            "proposal_size": proposal_size,
            "objective_metric": HPO_OBJECTIVE_METRIC,
            "objective_value": float(objective_value),
            "reference_p0": float(p0_reference),
            **summary_row,
        }
        trial_rows.append(trial_payload)
        for key, value in trial_payload.items():
            if isinstance(value, (str, int, float)) and not isinstance(value, bool):
                trial.set_user_attr(str(key), value)
        return float(objective_value)

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=BASE_SEED + 10_000 * scenario_idx + 7),
        study_name=f"{scenario.key}_oracle_beta",
    )
    study.optimize(objective, n_trials=HPO_N_TRIALS, show_progress_bar=False)

    best_multiplier = float(study.best_trial.params["beta_multiplier"])
    best_proposal_size = int(study.best_trial.params["proposal_size"])
    best_beta = float(beta_init * best_multiplier)
    best_config = {
        "scenario": scenario.key,
        "reference_p0": float(p0_reference),
        "beta_init": float(beta_init),
        "beta0_formula": float(init_payload["beta0_formula"]),
        "beta0_laplace": float(init_payload["beta0_laplace"]),
        "sigma_t": float(sigma_t),
        "best_beta_multiplier": best_multiplier,
        "best_beta": best_beta,
        "best_proposal_size": int(best_proposal_size),
        "best_objective_value": float(study.best_value),
        "best_trial_number": int(study.best_trial.number),
        "n_trials": int(len(study.trials)),
    }

    final_eval = run_named_mcmc_checkpoint_study(
        scenario.problem,
        scenario.exact_p,
        config_specs=[
            {
                "label": "init",
                "config_id": "init",
                "beta": float(beta_init),
                "proposal_size": int(baseline_swap),
                "source": "pilot_init",
            },
            {
                "label": "oracle",
                "config_id": "oracle",
                "beta": float(best_beta),
                "proposal_size": int(best_proposal_size),
                "source": "oracle_hpo",
                "selected_by_objectives": ["optuna"],
            },
        ],
        sigma_t=sigma_t,
        estimation_points=ESTIMATION_POINTS,
        repeats=FINAL_EVAL_REPEATS,
        base_seed=BASE_SEED + 5_000_000 + 100_000 * scenario_idx,
        template_cfg=final_eval_cfg,
        n_jobs=FINAL_EVAL_N_JOBS,
    )

    oracle_results[scenario.key] = {
        "best_config": best_config,
        "trial_rows": list(trial_rows),
        "final_eval": final_eval,
        "init_payload": {
            key: value
            for key, value in init_payload.items()
            if key != "pilot_t"
        },
    }

    if SAVE_OUTPUTS and run_dir is not None:
        scenario_dir = run_dir / scenario.key
        scenario_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(trial_rows).to_json(
            scenario_dir / "hpo_trials.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(final_eval["records"]).to_json(
            scenario_dir / "final_eval_records.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(final_eval["summary"]).to_json(
            scenario_dir / "final_eval_summary.json",
            orient="records",
            indent=2,
        )
        (scenario_dir / "best_config.json").write_text(
            json.dumps(best_config, indent=2),
            encoding="utf-8",
        )
        (scenario_dir / "metadata.json").write_text(
            json.dumps(
                {
                    "scenario": scenario.key,
                    "scenario_display": scenario.description,
                    "exact_p": float(scenario.exact_p),
                    "known_significance_threshold": scenario.extra.get("known_significance_threshold"),
                    "application_setting_key": scenario.extra.get("application_setting_key"),
                    "reference_p0": float(p0_reference),
                    "init_payload": oracle_results[scenario.key]["init_payload"],
                    "hpo_settings": {
                        "n_trials": int(HPO_N_TRIALS),
                        "trial_repeats": int(HPO_TRIAL_REPEATS),
                        "objective_metric": str(HPO_OBJECTIVE_METRIC),
                        "beta_multiplier_low": float(HPO_BETA_MULTIPLIER_LOW),
                        "beta_multiplier_high": float(HPO_BETA_MULTIPLIER_HIGH),
                        "swap_choices": [int(v) for v in swap_choices],
                        "checkpoint": int(HPO_CHECKPOINT),
                    },
                    "final_eval_settings": final_eval["settings"],
                },
                indent=2,
            ),
            encoding="utf-8",
        )

    best_row = pd.DataFrame([best_config])
    final_summary_df = pd.DataFrame(final_eval["summary"]).sort_values(["checkpoint", "label"])
    display(best_row)
    display(final_summary_df[[
        "checkpoint",
        "label",
        "mean_estimate",
        "rmse",
        "mean_abs_log10_error",
        "mean_variance_estimate",
        "mean_q_tilt_tail_share",
        "mean_ess",
        "mean_acceptance_rate",
        "mean_weight_cv",
    ]])

## Review Saved Outputs

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    display(pd.DataFrame([
        {
            "scenario": key,
            "best_beta": payload["best_config"]["best_beta"],
            "best_multiplier": payload["best_config"]["best_beta_multiplier"],
            "best_proposal_size": payload["best_config"]["best_proposal_size"],
            "best_objective_value": payload["best_config"]["best_objective_value"],
        }
        for key, payload in oracle_results.items()
    ]))
else:
    print("SAVE_OUTPUTS=False, so no saved outputs to display.")

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_BETA_DIR = None
# # Example:
# # RELOAD_BETA_DIR = project_root / "results" / "mcmcis_beta_notebook" / "20260411_120000_beta_diag" / "gwas_additive_score_sig_n60"

# if RELOAD_BETA_DIR is not None:
#     print(json.loads((RELOAD_BETA_DIR / "best_config.json").read_text()))
#     display(pd.read_json(RELOAD_BETA_DIR / "final_eval_summary.json"))
#     display(pd.read_json(RELOAD_BETA_DIR / "hpo_trials.jsonl", lines=True).sort_values("objective_value").head())
# else:
#     print("Set RELOAD_BETA_DIR to a saved oracle-beta directory to inspect saved results from disk only.")